# Text to Text Evaluation (Summarization, QA, Translation, Paraphrase, Style Transfer)

Evaluating text transformation tasks with ROUGE, BERTScore, SacreBLEU, ChrF, TER, readability metrics, semantic similarity, and LLM judges.

In [1]:
# !pip install sacrebleu evaluate bert-score sentence-transformers textstat

In [2]:
GROQ_MODEL_FAST  = "llama-3.1-8b-instant"
GROQ_MODEL_SMART = "llama-3.3-70b-versatile"
OPENAI_BASE_URL = "https://api.groq.com/openai/v1"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt',   quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ All packages installed!')

import os
import json, re
import numpy as np
from collections import Counter
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def groq_chat(prompt, system="You are a helpful assistant.",
              model=GROQ_MODEL_SMART, temperature=0.0, max_tokens=600):
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    raw = re.sub(r'```json|```', '', raw).strip()
    try: return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

def show(title, d):
    print(f"\n{'='*60}\n  📊 {title}\n{'='*60}")
    for k, v in d.items():
        print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")

print('🤖 Groq test:', groq_chat('Reply with only: Groq is ready!', max_tokens=20))

✅ All packages installed!
🤖 Groq test: Groq is ready!


## 1. Summarization (ROUGE + BERTScore + Faithfulness + Compression)
### Tools: HuggingFace Evaluate, Groq (faithfulness judge)

In [4]:
import evaluate

source = '''Scientists at MIT have developed a battery that charges smartphones in 5 minutes
while lasting 3x longer than current lithium-ion batteries. The breakthrough uses a novel
solid-state electrolyte that eliminates overheating risks. Commercial availability is
expected in 2-3 years. This could also transform the EV industry, reducing charging times
from hours to minutes while extending driving range significantly.'''

reference_summary = 'MIT developed a 5-minute-charge battery using solid-state electrolyte that lasts 3x longer, with commercial release in 2-3 years.'

generated_summaries = [
    ('Good Summary',     'MIT researchers created a battery that charges phones in 5 minutes and lasts 3x longer using solid-state electrolyte. Commercial release in 2-3 years.'),
    ('Vague Summary',    'Scientists made a new type of battery. It is very good and will help people in the future.'),
    ('Hallucinated',     'Harvard scientists developed a solar-powered battery using Bluetooth that charges cars wirelessly.'),
    ('Partial Summary',  'A new battery charges in 5 minutes.'),
    ('Over-detailed',    'MIT scientists have created a revolutionary battery using solid-state electrolyte technology that charges smartphones in exactly 5 minutes, lasts precisely 3 times longer than lithium-ion batteries, completely eliminates overheating risks, and will be commercially available in 2-3 years, with potential to transform the EV industry by reducing charging times and extending driving range.'),
]

rouge_m = evaluate.load('rouge')
bs_m    = evaluate.load('bertscore')

FAITH_SYS = '''Check if every claim in the summary is supported by the source document.
Return JSON only: {"faithful": true/false, "score": 0.0-1.0, "unsupported_claims": ["..."]}'''

print('\n📝 Summarization Evaluation')
print('='*70)
summ_results = []
for label, summary in generated_summaries:
    rr   = rouge_m.compute(predictions=[summary], references=[reference_summary])
    bsr  = bs_m.compute(predictions=[summary], references=[reference_summary], model_type='distilbert-base-uncased')
    
    # Compression ratio
    cr = len(summary.split()) / len(source.split())
    
    # Faithfulness (LLM judge)
    faith = parse_json(groq_chat(f'Source:\n{source}\n\nSummary:\n{summary}', system=FAITH_SYS))
    fi = '✅' if faith.get('faithful') else '❌'
    
    # Coverage: what fraction of source sentences are represented
    source_sents = [s.strip() for s in source.split('.') if s.strip()]
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer('all-MiniLM-L6-v2')
    from sklearn.metrics.pairwise import cosine_similarity as cs
    src_emb = st.encode(source_sents)
    sum_emb = st.encode([summary])
    sims = cs(sum_emb, src_emb)[0]
    coverage = np.mean(sims > 0.3)
    
    result = {
        'label': label, 'rouge1': rr['rouge1'], 'rouge2': rr['rouge2'], 'rougeL': rr['rougeL'],
        'bertscore_f1': float(np.mean(bsr['f1'])), 'compression': cr,
        'faithful': faith.get('faithful', False), 'faith_score': faith.get('score', 0),
        'coverage': coverage
    }
    summ_results.append(result)
    
    print(f'\n  [{label}]')
    print(f'  {summary[:90]}')
    print(f'  ROUGE-1={rr["rouge1"]:.4f}  ROUGE-2={rr["rouge2"]:.4f}  ROUGE-L={rr["rougeL"]:.4f}')
    print(f'  BERTScore-F1={float(np.mean(bsr["f1"])):.4f}  Compression={cr:.2f}  Coverage={coverage:.2f}')
    print(f'  {fi} Faithful={faith.get("faithful")}  score={faith.get("score",0):.2f}')
    if faith.get('unsupported_claims'):
        print(f'     Unsupported: {faith["unsupported_claims"]}')


📝 Summarization Evaluation


/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(



  [Good Summary]
  MIT researchers created a battery that charges phones in 5 minutes and lasts 3x longer usi
  ROUGE-1=0.7660  ROUGE-2=0.4444  ROUGE-L=0.5532
  BERTScore-F1=0.9296  Compression=0.40  Coverage=0.75
  ✅ Faithful=True  score=1.00


/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



  [Vague Summary]
  Scientists made a new type of battery. It is very good and will help people in the future.
  ROUGE-1=0.1500  ROUGE-2=0.0000  ROUGE-L=0.1500
  BERTScore-F1=0.7584  Compression=0.31  Coverage=0.75
  ❌ Faithful=False  score=0.20
     Unsupported: ['It is very good', 'will help people in the future']


...


  [Hallucinated]
  Harvard scientists developed a solar-powered battery using Bluetooth that charges cars wir
  ROUGE-1=0.2857  ROUGE-2=0.1212  ROUGE-L=0.2857
  BERTScore-F1=0.8025  Compression=0.21  Coverage=0.50
  ❌ Faithful=False  score=0.00
     Unsupported: ['Harvard scientists', 'solar-powered battery', 'using Bluetooth', 'charges cars wirelessly', 'wireless charging']


/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



  [Partial Summary]
  A new battery charges in 5 minutes.
  ROUGE-1=0.2759  ROUGE-2=0.0000  ROUGE-L=0.2069
  BERTScore-F1=0.7854  Compression=0.12  Coverage=0.50
  ✅ Faithful=True  score=1.00


/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



  [Over-detailed]
  MIT scientists have created a revolutionary battery using solid-state electrolyte technolo
  ROUGE-1=0.4103  ROUGE-2=0.1842  ROUGE-L=0.3590
  BERTScore-F1=0.8648  Compression=0.91  Coverage=0.75
  ✅ Faithful=True  score=1.00


## 2. QA (Exact Match + Token F1 — SQuAD-style)
### Tool: Custom metrics (standard in SQuAD / TriviaQA evals)

In [5]:
def token_f1(pred, gold):
    pt, gt = pred.lower().split(), gold.lower().split()
    common = sum((Counter(pt) & Counter(gt)).values())
    if not common: return 0.0
    pr = common/len(pt); rc = common/len(gt)
    return 2*pr*rc/(pr+rc)

def normalize_answer(s):
    """Normalize answer for comparison: lowercase, strip articles, punctuation."""
    s = s.lower().strip()
    s = re.sub(r'\b(a|an|the)\b', '', s)
    s = re.sub(r'[^a-z0-9\s]', '', s)
    return ' '.join(s.split())

qa_examples = [
    {'q': 'When was Python created?',          'pred': 'Python was created in 1991.',           'gold': '1991'},
    {'q': 'Who wrote Hamlet?',                  'pred': 'William Shakespeare wrote Hamlet.',     'gold': 'William Shakespeare'},
    {'q': 'Boiling point of water?',            'pred': 'Water boils at 100 degrees Celsius.',  'gold': '100 degrees Celsius'},
    {'q': 'Capital of Japan?',                  'pred': 'Osaka',                                 'gold': 'Tokyo'},
    {'q': 'How many planets in solar system?',  'pred': 'There are 8 planets.',                  'gold': '8'},
    {'q': 'Who painted the Mona Lisa?',         'pred': 'Leonardo da Vinci.',                    'gold': 'Leonardo da Vinci'},
    {'q': 'What is the speed of light?',        'pred': 'About 300,000 km/s.',                   'gold': '299,792 km/s'},
    {'q': 'Largest ocean?',                     'pred': 'The Pacific Ocean.',                     'gold': 'Pacific Ocean'},
]

em_list, f1_list, norm_em_list = [], [], []
print('\n❓ QA Evaluation — Exact Match + Token F1')
print('='*65)
for qa in qa_examples:
    em = 1.0 if qa['pred'].lower().strip()==qa['gold'].lower().strip() else 0.0
    norm_em = 1.0 if normalize_answer(qa['pred'])==normalize_answer(qa['gold']) else 0.0
    f1 = token_f1(qa['pred'], qa['gold'])
    em_list.append(em); f1_list.append(f1); norm_em_list.append(norm_em)
    icon = '✅' if em else ('⚠️' if f1 > 0.3 else '❌')
    print(f'  {icon}  Q: {qa["q"]}')
    print(f'     Pred={qa["pred"]:<35}  Gold={qa["gold"]:<20}  EM={em:.0f}  NormEM={norm_em:.0f}  F1={f1:.3f}')

print(f'\n  Avg Exact Match:      {np.mean(em_list):.4f}')
print(f'  Avg Normalized EM:    {np.mean(norm_em_list):.4f}')
print(f'  Avg Token F1:         {np.mean(f1_list):.4f}')


❓ QA Evaluation — Exact Match + Token F1
  ❌  Q: When was Python created?
     Pred=Python was created in 1991.          Gold=1991                  EM=0  NormEM=0  F1=0.000
  ⚠️  Q: Who wrote Hamlet?
     Pred=William Shakespeare wrote Hamlet.    Gold=William Shakespeare   EM=0  NormEM=0  F1=0.667
  ⚠️  Q: Boiling point of water?
     Pred=Water boils at 100 degrees Celsius.  Gold=100 degrees Celsius   EM=0  NormEM=0  F1=0.444
  ❌  Q: Capital of Japan?
     Pred=Osaka                                Gold=Tokyo                 EM=0  NormEM=0  F1=0.000
  ⚠️  Q: How many planets in solar system?
     Pred=There are 8 planets.                 Gold=8                     EM=0  NormEM=0  F1=0.400
  ⚠️  Q: Who painted the Mona Lisa?
     Pred=Leonardo da Vinci.                   Gold=Leonardo da Vinci     EM=0  NormEM=1  F1=0.667
  ❌  Q: What is the speed of light?
     Pred=About 300,000 km/s.                  Gold=299,792 km/s          EM=0  NormEM=0  F1=0.000
  ⚠️  Q: Largest ocean?
     Pr

## 3. Translation (SacreBLEU + ChrF + TER)
### Tool: sacrebleu

In [6]:
import sacrebleu

hypotheses = [
    'The weather is nice today.',
    'I am going to the store to buy some bread.',
    'She speaks three languages fluently.',
    'The cat is sleeping on the sofa.',
    'He finished the project ahead of schedule.',
]
references = [
    ['The weather is good today.',       'It is a beautiful day today.'],
    ['I am going to the shop for bread.','I will go to the store to get bread.'],
    ['She is fluent in three languages.','She speaks three languages well.'],
    ['The cat is resting on the couch.', 'A cat sleeps on the sofa.'],
    ['He completed the project early.',  'The project was finished before the deadline.'],
]

bleu_s = sacrebleu.corpus_bleu(hypotheses, list(zip(*references)))
chrf_s = sacrebleu.corpus_chrf(hypotheses, list(zip(*references)))
ter_s  = sacrebleu.corpus_ter(hypotheses,  list(zip(*references)))

print('\n🌍 Translation Evaluation')
print('='*60)
print(f'  SacreBLEU  : {bleu_s.score:.2f}  (n-gram precision based, higher=better)')
print(f'  ChrF       : {chrf_s.score:.2f}  (character n-gram F-score, higher=better)')
print(f'  TER        : {ter_s.score:.2f}   (edit rate, lower=better)')

# Signature includes details
print(f'\n  BLEU signature: {bleu_s.format()}')

print('\n  Per-sentence BLEU:')
for hyp, refs in zip(hypotheses, references):
    s = sacrebleu.sentence_bleu(hyp, refs)
    c = sacrebleu.sentence_chrf(hyp, refs)
    print(f'    BLEU={s.score:6.2f}  ChrF={c.score:6.2f}  ->  {hyp}')


🌍 Translation Evaluation
  SacreBLEU  : 43.28  (n-gram precision based, higher=better)
  ChrF       : 60.61  (character n-gram F-score, higher=better)
  TER        : 37.50   (edit rate, lower=better)

  BLEU signature: BLEU = 43.28 79.5/55.9/37.9/20.8 (BP = 1.000 ratio = 1.026 hyp_len = 39 ref_len = 38)

  Per-sentence BLEU:
    BLEU= 37.99  ChrF= 65.63  ->  The weather is nice today.
    BLEU= 58.77  ChrF= 62.04  ->  I am going to the store to buy some bread.
    BLEU= 53.73  ChrF= 80.17  ->  She speaks three languages fluently.
    BLEU= 50.00  ChrF= 57.90  ->  The cat is sleeping on the sofa.
    BLEU= 13.89  ChrF= 44.05  ->  He finished the project ahead of schedule.


## 4. Paraphrase Evaluation
### Tool: Sentence-Transformers (semantic similarity) + lexical divergence

In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st_model = SentenceTransformer('all-MiniLM-L6-v2')

def lexical_divergence(text1, text2):
    """Measure how different the surface forms are (higher = more different wording)."""
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    overlap = len(words1 & words2)
    total = len(words1 | words2)
    return 1 - (overlap / total) if total else 0

def paraphrase_quality(original, paraphrase):
    """Good paraphrase = high semantic similarity + high lexical divergence."""
    sem_sim = cosine_similarity(st_model.encode([original]), st_model.encode([paraphrase]))[0][0]
    lex_div = lexical_divergence(original, paraphrase)
    # Combined score: both semantic preservation AND surface diversity
    quality = (float(sem_sim) + lex_div) / 2
    return {'semantic_similarity': float(sem_sim), 'lexical_divergence': lex_div, 'paraphrase_quality': quality}

paraphrase_pairs = [
    ("The cat sat on the mat.", "A feline was resting upon the rug."),           # good paraphrase
    ("The cat sat on the mat.", "The cat sat on the mat."),                       # copy (bad)
    ("The cat sat on the mat.", "The weather is beautiful today."),               # unrelated (bad)
    ("Machine learning enables computers to learn from data.", 
     "Computers can acquire knowledge from datasets through machine learning."),  # good
    ("The stock market crashed yesterday.", 
     "Yesterday the stock market crashed."),                                       # trivial reorder
]

print('\n📝 Paraphrase Quality Evaluation')
print('='*70)
for orig, para in paraphrase_pairs:
    scores = paraphrase_quality(orig, para)
    sem = scores['semantic_similarity']
    lex = scores['lexical_divergence']
    qual = scores['paraphrase_quality']
    
    if sem > 0.7 and lex > 0.3:
        label = "✅ Good paraphrase"
    elif sem > 0.7 and lex < 0.15:
        label = "⚠️ Near copy"
    elif sem < 0.4:
        label = "❌ Not a paraphrase"
    else:
        label = "⚠️ Mediocre"
    
    print(f'\n  {label}')
    print(f'    Original:   {orig[:60]}')
    print(f'    Paraphrase: {para[:60]}')
    print(f'    Semantic={sem:.3f}  Lexical Div={lex:.3f}  Quality={qual:.3f}')

/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



📝 Paraphrase Quality Evaluation

  ⚠️ Mediocre
    Original:   The cat sat on the mat.
    Paraphrase: A feline was resting upon the rug.
    Semantic=0.559  Lexical Div=0.909  Quality=0.734

  ⚠️ Near copy
    Original:   The cat sat on the mat.
    Paraphrase: The cat sat on the mat.
    Semantic=1.000  Lexical Div=0.000  Quality=0.500

  ❌ Not a paraphrase
    Original:   The cat sat on the mat.
    Paraphrase: The weather is beautiful today.
    Semantic=-0.008  Lexical Div=0.889  Quality=0.440

  ✅ Good paraphrase
    Original:   Machine learning enables computers to learn from data.
    Paraphrase: Computers can acquire knowledge from datasets through machin
    Semantic=0.835  Lexical Div=0.786  Quality=0.810

  ✅ Good paraphrase
    Original:   The stock market crashed yesterday.
    Paraphrase: Yesterday the stock market crashed.
    Semantic=0.986  Lexical Div=0.571  Quality=0.779


## 5. Groq End-to-End Paraphrase Generation + Evaluation
### Tool: Groq for paraphrase generation, then evaluate quality

In [8]:
PARA_SYS = "Paraphrase the following text. Use different words and sentence structure while preserving the meaning. Return only the paraphrase."

originals = [
    "Artificial intelligence is transforming the healthcare industry by enabling faster diagnosis and personalized treatment.",
    "The Great Barrier Reef is the largest coral reef system in the world, located off the coast of Australia.",
    "Quantum computing has the potential to solve problems that are impossible for classical computers.",
    "Climate change is causing sea levels to rise, threatening coastal communities worldwide.",
]

print('\n🤖 Groq Paraphrase Generation + Evaluation')
print('='*70)
para_quality_scores = []
for orig in originals:
    para = groq_chat(orig, system=PARA_SYS, model=GROQ_MODEL_FAST, max_tokens=150)
    scores = paraphrase_quality(orig, para)
    para_quality_scores.append(scores['paraphrase_quality'])
    
    sem = scores['semantic_similarity']
    lex = scores['lexical_divergence']
    print(f'\n  Original:   {orig[:65]}')
    print(f'  Paraphrase: {para[:65]}')
    print(f'  Semantic={sem:.3f}  Lexical Div={lex:.3f}  Quality={scores["paraphrase_quality"]:.3f}')

print(f'\n  Average Paraphrase Quality: {np.mean(para_quality_scores):.4f}')


🤖 Groq Paraphrase Generation + Evaluation

  Original:   Artificial intelligence is transforming the healthcare industry b
  Paraphrase: The healthcare sector is undergoing a significant transformation 
  Semantic=0.821  Lexical Div=0.844  Quality=0.832

  Original:   The Great Barrier Reef is the largest coral reef system in the wo
  Paraphrase: Stretching along the Australian coastline, the Great Barrier Reef
  Semantic=0.846  Lexical Div=0.739  Quality=0.793

  Original:   Quantum computing has the potential to solve problems that are im
  Paraphrase: Classical computers face significant limitations when tackling co
  Semantic=0.802  Lexical Div=0.889  Quality=0.845

  Original:   Climate change is causing sea levels to rise, threatening coastal
  Paraphrase: Rising global temperatures are having a profound impact on the wo
  Semantic=0.805  Lexical Div=0.848  Quality=0.827

  Average Paraphrase Quality: 0.8243


## 6. Style Transfer Evaluation
### Tool: Groq (style transfer) + semantic preservation + style classifier

In [9]:
STYLE_SYS_FORMAL = "Rewrite the following casual/informal text in a formal, professional tone. Return only the rewritten text."
STYLE_SYS_CASUAL = "Rewrite the following formal text in a casual, conversational tone. Return only the rewritten text."

STYLE_JUDGE_SYS = '''Evaluate the style transfer. Score each dimension (0.0-1.0):
1. style_accuracy: Does it match the target style?
2. content_preservation: Is the meaning preserved?
3. fluency: Is it grammatically correct and natural?
Return JSON only: {"style_accuracy":..., "content_preservation":..., "fluency":...}'''

style_tests = [
    {"text": "Hey, this AI stuff is pretty cool and it's gonna change everything lol",
     "target": "formal", "system": STYLE_SYS_FORMAL},
    {"text": "The implementation of artificial intelligence systems necessitates careful consideration of ethical implications.",
     "target": "casual", "system": STYLE_SYS_CASUAL},
    {"text": "yo the new phone is fire, battery lasts forever and the camera slaps",
     "target": "formal", "system": STYLE_SYS_FORMAL},
]

print('\n🎭 Style Transfer Evaluation')
print('='*70)
style_scores = []
for test in style_tests:
    transferred = groq_chat(test["text"], system=test["system"], model=GROQ_MODEL_FAST, max_tokens=150)
    
    # Semantic preservation
    sem = cosine_similarity(st_model.encode([test["text"]]), st_model.encode([transferred]))[0][0]
    
    # Style quality (LLM judge)
    judge_prompt = f'Original ({"casual" if test["target"]=="formal" else "formal"}): {test["text"]}\nTransferred ({test["target"]}): {transferred}'
    judge = parse_json(groq_chat(judge_prompt, system=STYLE_JUDGE_SYS))
    
    sa = judge.get('style_accuracy', 0)
    cp = judge.get('content_preservation', 0)
    fl = judge.get('fluency', 0)
    style_scores.append({'sa': sa, 'cp': cp, 'fl': fl, 'sem': float(sem)})
    
    print(f'\n  Target: {test["target"]}')
    print(f'  Input:  {test["text"][:65]}')
    print(f'  Output: {transferred[:65]}')
    print(f'  Style Acc={sa:.2f}  Content Pres={cp:.2f}  Fluency={fl:.2f}  Semantic={float(sem):.3f}')

avg_sa = np.mean([s['sa'] for s in style_scores])
avg_cp = np.mean([s['cp'] for s in style_scores])
print(f'\n  Avg Style Accuracy: {avg_sa:.3f}')
print(f'  Avg Content Preservation: {avg_cp:.3f}')


🎭 Style Transfer Evaluation

  Target: formal
  Input:  Hey, this AI stuff is pretty cool and it's gonna change everythin
  Output: The advent of artificial intelligence technology is a significant
  Style Acc=0.90  Content Pres=0.80  Fluency=0.90  Semantic=0.479

  Target: casual
  Input:  The implementation of artificial intelligence systems necessitate
  Output: We need to think really carefully about the ethics of AI systems 
  Style Acc=0.90  Content Pres=0.95  Fluency=0.90  Semantic=0.789

  Target: formal
  Input:  yo the new phone is fire, battery lasts forever and the camera sl
  Output: The newly released smartphone has demonstrated exceptional perfor
  Style Acc=0.90  Content Pres=0.90  Fluency=0.95  Semantic=0.591

  Avg Style Accuracy: 0.900
  Avg Content Preservation: 0.883


## 7. Text Simplification (Readability Metrics)
### Tool: textstat for readability + Groq for simplification

In [10]:
import textstat

SIMPLIFY_SYS = "Simplify the following text so a 10-year-old can understand it. Use short sentences and simple words. Return only the simplified text."

complex_texts = [
    "The ramifications of anthropogenic climate change are multifaceted, encompassing alterations in precipitation patterns, exacerbation of extreme weather events, and perturbations to marine ecosystems through ocean acidification.",
    "Quantum entanglement, a phenomenon whereby two particles become intrinsically linked such that the quantum state of one instantaneously influences the other regardless of spatial separation, challenges classical notions of locality.",
    "The mitochondrial electron transport chain facilitates oxidative phosphorylation through a series of redox reactions, culminating in the chemiosmotic generation of adenosine triphosphate.",
]

print('\n📖 Text Simplification Evaluation')
print('='*70)
simplification_results = []
for text in complex_texts:
    simple = groq_chat(text, system=SIMPLIFY_SYS, model=GROQ_MODEL_FAST, max_tokens=200)
    
    # Readability scores (before and after)
    orig_fk = textstat.flesch_kincaid_grade(text)
    simp_fk = textstat.flesch_kincaid_grade(simple)
    orig_fre = textstat.flesch_reading_ease(text)
    simp_fre = textstat.flesch_reading_ease(simple)
    orig_ari = textstat.automated_readability_index(text)
    simp_ari = textstat.automated_readability_index(simple)
    orig_dc = textstat.dale_chall_readability_score(text)
    simp_dc = textstat.dale_chall_readability_score(simple)
    
    # Semantic preservation
    sem = cosine_similarity(st_model.encode([text]), st_model.encode([simple]))[0][0]
    
    # Word-level simplification
    orig_words = text.split()
    simp_words = simple.split()
    avg_word_len_orig = np.mean([len(w) for w in orig_words])
    avg_word_len_simp = np.mean([len(w) for w in simp_words])
    
    simplification_results.append({
        'fk_drop': orig_fk - simp_fk, 'semantic': float(sem),
        'fre_gain': simp_fre - orig_fre,
    })
    
    print(f'\n  Original: {text[:65]}...')
    print(f'  Simple:   {simple[:65]}...')
    print(f'  {"Metric":<30} {"Original":>10} {"Simplified":>12} {"Change":>10}')
    print(f'  {"-"*62}')
    print(f'  {"Flesch-Kincaid Grade":<30} {orig_fk:>10.1f} {simp_fk:>12.1f} {simp_fk-orig_fk:>+10.1f}')
    print(f'  {"Flesch Reading Ease":<30} {orig_fre:>10.1f} {simp_fre:>12.1f} {simp_fre-orig_fre:>+10.1f}')
    print(f'  {"Automated Readability":<30} {orig_ari:>10.1f} {simp_ari:>12.1f} {simp_ari-orig_ari:>+10.1f}')
    print(f'  {"Dale-Chall":<30} {orig_dc:>10.1f} {simp_dc:>12.1f} {simp_dc-orig_dc:>+10.1f}')
    print(f'  {"Avg Word Length":<30} {avg_word_len_orig:>10.1f} {avg_word_len_simp:>12.1f} {avg_word_len_simp-avg_word_len_orig:>+10.1f}')
    print(f'  {"Semantic Preservation":<30} {"":>10} {float(sem):>12.3f}')

avg_fk_drop = np.mean([r['fk_drop'] for r in simplification_results])
avg_sem = np.mean([r['semantic'] for r in simplification_results])
print(f'\n  Avg Grade Level Reduction: {avg_fk_drop:.1f} grades')
print(f'  Avg Semantic Preservation: {avg_sem:.3f}')


📖 Text Simplification Evaluation

  Original: The ramifications of anthropogenic climate change are multifacete...
  Simple:   The Earth is getting sick because of things people are doing. 

-...
  Metric                           Original   Simplified     Change
  --------------------------------------------------------------
  Flesch-Kincaid Grade                 26.3          5.1      -21.2
  Flesch Reading Ease                 -47.3         77.3     +124.7
  Automated Readability                28.2          4.6      -23.5
  Dale-Chall                           14.0          8.3       -5.7
  Avg Word Length                       7.8          4.4       -3.3
  Semantic Preservation                            0.470

  Original: Quantum entanglement, a phenomenon whereby two particles become i...
  Simple:   Imagine you have two toy cars that are connected in a special way...
  Metric                           Original   Simplified     Change
  ----------------------------------------

## 8. Aggregated Text-to-Text Scorecard

In [11]:
best_summ = max(summ_results, key=lambda x: x['rougeL'])

show('Text-to-Text Evaluation Scorecard', {
    'Summarization ROUGE-L (best)':    best_summ['rougeL'],
    'Summarization BERTScore (best)':  best_summ['bertscore_f1'],
    'QA Exact Match':                  float(np.mean(em_list)),
    'QA Normalized EM':                float(np.mean(norm_em_list)),
    'QA Token F1':                     float(np.mean(f1_list)),
    'Translation SacreBLEU':           bleu_s.score / 100,
    'Translation ChrF':                chrf_s.score / 100,
    'Translation TER (↓ better)':      ter_s.score / 100,
    'Paraphrase Quality':              float(np.mean(para_quality_scores)),
    'Style Transfer Accuracy':         avg_sa,
    'Style Content Preservation':      avg_cp,
    'Simplification Grade Drop':       avg_fk_drop,
    'Simplification Semantic Pres':    avg_sem,
})


  📊 Text-to-Text Evaluation Scorecard
  Summarization ROUGE-L (best)  : 0.5532
  Summarization BERTScore (best): 0.9296
  QA Exact Match                : 0.0000
  QA Normalized EM              : 0.2500
  QA Token F1                   : 0.3222
  Translation SacreBLEU         : 0.4328
  Translation ChrF              : 0.6061
  Translation TER (↓ better)    : 0.3750
  Paraphrase Quality            : 0.8243
  Style Transfer Accuracy       : 0.9000
  Style Content Preservation    : 0.8833
  Simplification Grade Drop     : 16.1037
  Simplification Semantic Pres  : 0.5172
